#### Parallele und funktionale Programmierung
### Übungsblatt 11
---

## <u>Bonus</u>aufgabe 11.5: Zugverbindungen
Sie wollen mit dem Zug vom Bahnhof `s` zum Bahnhof `d` gelangen. Dafür wollen Sie höchstens `maxCost` Euro für Fahrtpreise ausgeben. Zudem möchten Sie möglichst wenig umsteigen.  
In dieser Aufgabe sollen Sie das beschriebene Problem funktional lösen. Dazu sind die Zugverbindungen als Adjazenzliste dargestellt (ein Eintrag besteht aus (Nachfolger, Kosten)):  
<img src="lib/p03_Graph_Zugverbindungen.png" width="225px"/>

In [1]:
val g = List(
   List((1, 1), (2, 3), (4, 10)),
   List((2, 1)),
   List((3, 1), (4, 5)),
   List((4, 1)),
   List())

g: List[List[(Int, Int)]] = List(
  List((1, 1), (2, 3), (4, 10)),
  List((2, 1)),
  List((3, 1), (4, 5)),
  List((4, 1)),
  List()
)

Implementieren Sie dazu die folgenden Funktionen in die Datei `Trains.scala`:

**a)** `enumPaths: (List[List[(Int, Int)]], List[(Int, Int)], Int, Int) => List[Int]`: Die Funktion `enumPaths(als, f, d, n)` erhält einen als Adjazenzliste `als` dargestellten Graphen, der die Zugverbindungen beschreibt, eine Liste bereits besuchter Knoten `f` mit den dazugehörigen Kosten (wieder als Tupel (Knoten, Kosten)), den Zielbahnhof `d` sowie die maximale Pfadlänge `n`. Damit berechnet `enumPaths` die Liste der Kosten aller Pfade, die von einem Knoten in `f` starten und über maximal `n` Kanten zu `d` führen. Die Reihenfolge der Kosten in der Ergebnisliste ist unerheblich. Verwenden Sie zur Lösung Listengeneratoren und Rekursion.  
**Beispiele:**  
`enumPaths(g, List((0, 0)), 2, 1) == List(3) // 0->2`  
`enumPaths(g, List((0, 0)), 2, 2) == List(2, 3) // 0->1->2, 0->2`

In [2]:
def enumPaths: (List[List[(Int, Int)]], List[(Int, Int)], Int, Int) => List[Int] = (als, rs, d, n) => {
    val costList = for (u <- rs if (u._1 == d)) yield u._2
    val g = for (x <- rs; y <- als(x._1)) yield (y._1, x._2 + y._2)
    if (n == 0) {
      costList
    } else {
      costList ::: enumPaths(als, g, d, n - 1)
    }
  }

defined function enumPaths

In [3]:
// Kleiner Testfall zu dieser Teilaufgabe:
{val f: (List[List[(Int, Int)]], List[(Int, Int)], Int, Int) => List[Int] = enumPaths}
assert(List(3) == enumPaths(g, List((0, 0)), 2, 1)); print("✅")
assert(Set(2, 3) == enumPaths(g, List((0, 0)), 2, 2).toSet); print("✅")

✅✅

**b)** `pathsWithMaxCost: (List[List[(Int, Int)]], Int, Int, Int) => Int => Boolean`: Die Funktion `pathsWithMaxCost(als, s, d, maxCost)(n)` prüft, ob es einen Pfad mit maximaler Länge `n` von `s` nach `d` im durch die Adjazenzliste `als` dargestellten Graphen gibt, dessen Kosten nicht größer als `maxCost` sind. Verwenden Sie dazu die Funktion `enumPaths` und eine Filteroperation.  
**Beispiele:**  
`pathsWithMaxCost(g, 0, 2, 3)(1) == true`  
`pathsWithMaxCost(g, 0, 2, 2)(1) == false`

In [4]:
def pathsWithMaxCost: (List[List[(Int, Int)]], Int, Int, Int) => Int => Boolean = (als, s, d, c) => n => {
  // enumPaths(als, List((s, 0)), d, n).exists(_ <= c)
  enumPaths(als, List((s, 0)), d, n).filter(_ <= c).size > 0
}

defined function pathsWithMaxCost

In [5]:
// Kleiner Testfall zu dieser Teilaufgabe:
{val f: (List[List[(Int, Int)]], Int, Int, Int) => Int => Boolean = pathsWithMaxCost}
assert(pathsWithMaxCost(g, 0, 2, 3)(1)); print("✅")
assert(!pathsWithMaxCost(g, 0, 2, 2)(1)); print("✅")

✅✅

**c)** `binarySearch: (Int => Boolean) => (Int, Int) => Int`: Die Funktion `binarySearch(f)(l, h)` führt eine binäre Suche auf dem Intervall $[$ `l` $;$ `h` $[$ nach dem kleinsten Element $x \in [$ `l` $;$ `h` $[$ durch, für das `f`($x$) == `true` gilt. Wenn kein solches $x$ im Intervall liegt, soll `h` zurückgegeben werden.  
Somit lässt sich die minimale Anzahl an Umstiegen bestimmen als:  
`binarySearch(pathsWithMaxCost(g, s, d, maxCost))(0, g.length)`  
**Beispiele:**  
`binarySearch(pathsWithMaxCost(g, 0, 2, 3))(0, g.length) == 1`  
`binarySearch(pathsWithMaxCost(g, 0, 2, 2))(0, g.length) == 2`  
`binarySearch(pathsWithMaxCost(g, 0, 4, 8))(0, g.length) == 2`  
**Hinweis:** Sie können `binarySearch` unabhängig von den vorherigen Teilaufgaben testen:  
`fibs = List(0, 1, 1, 2, 3, 5, 8, 13)`  
`binarySearch(fibs(_) > 5)(0, fibs.length) == 6`

In [6]:
def binarySearch: (Int => Boolean) => (Int, Int) => Int = f => (l, h) => {
  val mitte = l + ((h - l) / 2)
  if (l >= h) {
    h
  } else if (f(mitte)) {
    binarySearch(f)(l, mitte)
  } else {
    binarySearch(f)(mitte+1, h)
  }
}

defined function binarySearch

In [7]:
// Kleiner Testfall zu dieser Teilaufgabe:
{val f: (Int => Boolean) => (Int, Int) => Int = binarySearch}
assert(1 == binarySearch(pathsWithMaxCost(g, 0, 2, 3))(0, g.length)); print("✅")
assert(2 == binarySearch(pathsWithMaxCost(g, 0, 2, 2))(0, g.length)); print("✅")
assert(2 == binarySearch(pathsWithMaxCost(g, 0, 4, 8))(0, g.length)); print("✅")
{
val fibs = List(0, 1, 1, 2, 3, 5, 8, 13)
assert(6 == binarySearch(fibs(_) > 5)(0, fibs.length)); print("✅")
}

✅✅✅✅